# 🦠 JORINOVA NEXUS — Stool protozoa CLASSIFIER (fine-tuning)

Classifies intestinal protozoa (Entamoeba histolytica, Giardia, Cryptosporidium) on stool
microscopy. This is a **classification** model (`yolov8*-cls`) — the ready public data (Kaggle
protozoan-parasite image set) is class-labelled, and classification reaches high accuracy
(metric = **top-1 accuracy**).

Produces `protozoa.pt`; the predicted class resolves to its disease via
`backend/ai_services/protozoa_organisms.json`.

> **Fine-tuning, NOT from scratch** (starts from `yolov8m-cls.pt`).
> **Data source: KAGGLE** — upload your `kaggle.json` in step 4.

**Before you start:** Runtime -> **Change runtime type** -> **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics kaggle
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_protozoa', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_protozoa')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data (KAGGLE — protozoan parasite image set)
Upload your `kaggle.json` (kaggle.com → Settings → API → **Create New Token**). The cell downloads
the **protozoan-parasite** set (Cryptosporidium / Entamoeba / Giardia), auto-detects the class
folders, and renames them to the app keys in `protozoa_organisms.json`.

In [ ]:
# ── 4. Protozoa data via KAGGLE (classification) -> train/val class folders ──
# Ready dataset: Cryptosporidium / Entamoeba histolytica / Giardia (~6.7k imgs).
# Class folders are auto-detected and renamed to the app keys in protozoa_organisms.json.
import os, glob, shutil, subprocess, sys, random
# 1) Kaggle token (kaggle.com -> Settings -> API -> Create New Token -> kaggle.json)
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Upload kaggle.json ...'); up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    name = 'kaggle.json' if os.path.exists('kaggle.json') else list(up)[0]
    shutil.move(name, '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

RAW = '/content/data/protozoa_raw'
if os.path.exists(RAW): shutil.rmtree(RAW)
os.makedirs(RAW, exist_ok=True)
SLUGS = ['ankushmallick/protozoan-parasite-image-dataset-augmented',
         'ankushmallick/protozoan-parasite-image-dataset']
ok = False
for DATASET in SLUGS:
    rc = subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET, '-p', RAW, '--unzip']).returncode
    if rc == 0: ok = True; print('downloaded:', DATASET); break
    print('failed:', DATASET)
assert ok, 'Kaggle download failed — accept the dataset rules on kaggle.com + check kaggle.json.'

# 2) canon-map folder names -> app keys (protozoa_organisms.json)
def canon(n):
    n = n.lower()
    if 'crypto' in n: return 'cryptosporidium'
    if 'entamoeba' in n or 'histolytica' in n or 'amoeb' in n: return 'entamoeba_histolytica'
    if 'giardia' in n: return 'giardia_lamblia'
    if 'cyclospora' in n: return 'cyclospora'
    if 'isospora' in n: return 'cystoisospora'
    return n.strip().replace(' ', '_').replace('-', '_')

exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff')
CLS = '/content/data/protozoa_cls'
if os.path.exists(CLS): shutil.rmtree(CLS)
n_imgs = 0
for d, subs, files in os.walk(RAW):
    imgs = [p for e in exts for p in glob.glob(os.path.join(d, e))]
    if not imgs: continue
    key = canon(os.path.basename(d))
    random.shuffle(imgs); nval = max(1, len(imgs) // 6)
    for i, ip in enumerate(imgs):
        split = 'val' if i < nval else 'train'
        out = os.path.join(CLS, split, key); os.makedirs(out, exist_ok=True)
        shutil.copy(ip, os.path.join(out, '%s_%d%s' % (key, n_imgs + i, os.path.splitext(ip)[1])))
    n_imgs += len(imgs)
assert n_imgs, 'No images found under ' + RAW + ' — print os.listdir(RAW) and send it to me.'

DATA_DIR = CLS
counts = {s: {c: len(os.listdir(os.path.join(CLS, s, c))) for c in sorted(os.listdir(os.path.join(CLS, s)))}
          for s in ('train', 'val') if os.path.isdir(os.path.join(CLS, s))}
print('DATA_DIR =', DATA_DIR)
print('classes:', sorted(os.listdir(os.path.join(CLS, 'train'))))
print('counts:', counts)

## 5. Choose base weights (fine-tune, never from scratch)

In [ ]:
BASE_WEIGHTS = 'yolov8m-cls.pt'   # classification model (yolov8s-cls = faster/less accurate)
print('fine-tuning (classification) from:', BASE_WEIGHTS)

## 6. Fine-tune the classifier
Metric is **top-1 accuracy** (not mAP). Protozoa cyst/oocyst crops → `imgsz=224`, `yolov8m-cls`,
~120 epochs, cosine LR + dropout/erasing. Microscopy has no canonical orientation → flips + rotation are safe.

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_protozoa/runs'
try: os.makedirs(PROJECT, exist_ok=True)
except Exception: PROJECT = '/content/runs/classify'
model = YOLO(BASE_WEIGHTS)   # yolov8m-cls -> classification (protozoa cyst/oocyst crops)
model.train(data=DATA_DIR, epochs=120, imgsz=224, batch=64,
            project=PROJECT, name='protozoa', pretrained=True, patience=30, plots=True,
            cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 6b. RESUME if a run disconnects (continue, no 4-vs-80 crash)

In [ ]:
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_protozoa/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_protozoa/**/*.pt', recursive=True)
                           + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet — run step 6 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = None
try:
    if ckpt in runs: model = YOLO(ckpt); model.train(resume=True)
    else: raise RuntimeError('loose')
except Exception as e:
    print('continue-from-weights:', str(e)[:80]); model = None
if model is None:
    assert 'DATA_DIR' in globals(), 'Re-run step 4 first so DATA_DIR is set.'
    model = YOLO(ckpt); model.train(data=DATA_DIR, epochs=120, imgsz=224, batch=64,
        project='/content/drive/MyDrive/nexus_protozoa/runs', name='protozoa_cont', pretrained=True, patience=30,
        cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 7. Export the model (Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = '/content/drive/MyDrive/nexus_protozoa'
best = sorted([c for c in glob.glob('/content/**/weights/best.pt', recursive=True)
               + glob.glob(DRIVE + '/**/weights/best.pt', recursive=True) if os.path.isfile(c)],
              key=os.path.getmtime)[-1]
print('using weights:', best)
from ultralytics import YOLO; print('classes:', YOLO(best).model.names)
try:
    os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/protozoa.pt'); print('Drive ->', DRIVE + '/protozoa.pt')
except Exception as e: print('drive copy skipped:', e)
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename the download to **`protozoa.pt`** at **`backend/models/protozoa/protozoa.pt`**, commit, push.
2. Auto-loads for `image_type` ∈ {`protozoa`, `stool_protozoa`}; the predicted class resolves via `protozoa_organisms.json`. The vision service **handles classification models automatically** (top-1 + top-k).
3. On Render, build with `INSTALL_ML=1` (>=2 GB); else Claude vision reads the field.

> ⚠️ Screening aid — a microscopist confirms (wet prep / trichrome / modified acid-fast).